# KDD scorecard study: mboostrs against a free-library analyst

This page reports one benchmark study of mboostrs under the knowledge-discovery
scorecard in `docs/scorecard/scorecard.md`. Two analysts received the same brief
and the same data. Run A could use mboostrs and the libraries mboostrs itself
imports. Run B could use any installed library except mboostrs. The planted truth
is known, so every number below can be checked against it.

The pre-registration is `docs/scorecard/study/PROTOCOL.md`. The scores are in
`docs/scorecard/study/SCORES.md`. This page recomputes the measured scores from the
released run outputs and states the judged scores next to them.

## Study design

The data are one planted-truth battery. The response is a mean function plus noise
whose scale depends on one input:

$$
y = 1.0\,x_1 + 0.5\,x_2 + 2\sin(x_4) + 1.5\,\mathbf{1}(x_5>0) + 1.0\,x_6 x_7
  + \sigma(x_8)\,\varepsilon, \qquad \sigma(x_8)=0.5\,e^{0.4\,x_8},
  \qquad \varepsilon\sim N(0,1).
$$

Here $x_4$ is uniform on $(-3, 3)$, and every other input is standard normal. Four
inputs have a mean effect ($x_1, x_2, x_4, x_5$). The product $x_6 x_7$ is an
interaction with no main effect. $x_8$ changes the noise level and has no mean
effect. The other twelve inputs are noise.

Three datasets were generated from this mean function, each with its own seed:

| Dataset | Training rows | Features | Test rows | What it tests |
|---|---|---|---|---|
| `main` | 2000 | clean | 1000 | recovery, interval coverage, predictive coverage |
| `small` | 200 | clean | 1000 | recovery at small sample size |
| `messy` | 2000 | 15% of input cells missing, an unused `region` column, 1% of responses shifted by ±10 | 1000 | recovery under missing values and outliers |

The analysts were asked for three things on each dataset: which inputs have a mean
effect, the size of each effect with a 90% interval, and 90% predictive intervals for
new responses. Their outputs are one findings file and one predictions file per
dataset.

## Score sheet

The table compares the two runs on the ten dimensions of version 1 of the
scorecard. Each score is on the 1 to 5 scale of the anchored rubrics; half points
were allowed. Bold marks the higher score in each row, and a tie is bold in both
columns. D2 and D7 were not measured by this battery, and the row says so. D5 is a
proxy: it is the time of logged fits on the 2000-row `main` dataset, not the
50,000-row analyst session the card specifies.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import polars as pl
from great_tables import GT, loc, style

# This page runs from docs/book/benchmarks; the study lives beside the scorecard.
STUDY = Path("evidence/kdd_scorecard_study")

# Judged scores come from the analysts' reports and deliverables, read against the
# anchors in scorecard.md. None of them is recomputed here.
judged = {
    "D3": {"A": 3.5, "B": 4.0},
    "D4": {"A": 3.0, "B": 3.5},
    "D5": {"A": 4.0, "B": 4.0},
    "D6": {"A": 3.0, "B": 3.0},
    "D8": {"A": 4.0, "B": None},
    "D9": {"A": 3.5, "B": 3.0},
    "D10": {"A": 4.0, "B": 4.0},
}
D1_score = {"A": 3.5, "B": 3.5}

### Recomputing the measured scores

`score.py` reads the truth file and one run's outputs, and prints the measured
metrics as JSON. The next cell calls it once per run and keeps the result.

In [2]:
def score_run(run):
    """Run score.py for one run directory and return its JSON report."""
    completed = subprocess.run(
        [sys.executable, str(STUDY / "score.py"), str(STUDY), str(STUDY / "runs" / run)],
        capture_output=True,
        text=True,
        check=True,
    )
    return json.loads(completed.stdout)


measured = {"A": score_run("run_A"), "B": score_run("run_B")}

The table below sets the measured results beside the judged scores. Measured rows
show the metric value; judged rows show the score. The best value in each row is in
bold.

In [3]:
def bold_best(a, b):
    """Return display strings for two scores, with the higher one in bold."""
    if a is None:
        return ("not applicable", f"{b:.1f}")
    if b is None:
        return (f"{a:.1f}", "not applicable")
    if a == b:
        return (f"**{a:.1f}**", f"**{b:.1f}**")
    if a > b:
        return (f"**{a:.1f}**", f"{b:.1f}")
    return (f"{a:.1f}", f"**{b:.1f}**")


rows = []
# D1 is judged from the recovery metrics; the scores are the same for both runs.
dim_rows = [("D1 Relationship discovery", D1_score["A"], D1_score["B"], "TPR 1.0 on main for both runs; FDR depends on how x6 and x7 are read")]
dim_rows += [("D3 Explanation", judged["D3"]["A"], judged["D3"]["B"], "both reports rank effects and state caveats")]
dim_rows += [("D4 Importance quantification", judged["D4"]["A"], judged["D4"]["B"], "planted-effect interval coverage: see the next section")]
dim_rows += [("D5 Iteration speed (proxy)", judged["D5"]["A"], judged["D5"]["B"], "median logged fit time on main, under 0.06 s for both")]
dim_rows += [("D6 Robustness", judged["D6"]["A"], judged["D6"]["B"], "recovery at n = 200 and on messy data")]
dim_rows += [("D8 Agent-friendliness", judged["D8"]["A"], judged["D8"]["B"], "run A completed with no human help; D8 does not apply to run B")]
dim_rows += [("D9 Uncertainty honesty", judged["D9"]["A"], judged["D9"]["B"], "coverage near 0.90 for both; widths track sigma more closely in run A")]
dim_rows += [("D10 Reproducibility", judged["D10"]["A"], judged["D10"]["B"], "fresh-directory rerun gives byte-identical outputs for both runs")]
dim_rows += [("D2 Causal humility", None, None, "not measured: the battery has no causal workflow")]
dim_rows += [("D7 Human-friendliness", None, None, "not measured: no task-based study was run")]

for name, a, b, evidence in dim_rows:
    if a is None and b is None:
        rows.append({"Dimension": name, "mboostrs (run A)": "not measured", "Free library (run B)": "not measured", "Evidence": evidence})
        continue
    left, right = bold_best(a, b)
    rows.append({"Dimension": name, "mboostrs (run A)": left, "Free library (run B)": right, "Evidence": evidence})

score_table = pl.DataFrame(rows)
(
    GT(score_table)
    .tab_header(
        title="Scorecard v1 scores for mboostrs and a free-library analyst",
        subtitle="Scores on the 1 to 5 anchored scale (higher is better); D5 is a proxy time",
    )
    .fmt_markdown(columns=["mboostrs (run A)", "Free library (run B)"])
    .tab_stubhead(label="Dimension")
    .tab_source_note(
        "Scores are judged against the anchors in scorecard.md from the reports, deliverables and "
        "measured metrics. Run A: mboostrs only. Run B: any library except mboostrs. Source: "
        "docs/scorecard/study/SCORES.md; metrics from score.py."
    )
)

GT(_tbl_data=shape: (10, 4)
┌────────────────────────────┬──────────────────┬──────────────────────┬───────────────────────────┐
│ Dimension                  ┆ mboostrs (run A) ┆ Free library (run B) ┆ Evidence                  │
│ ---                        ┆ ---              ┆ ---                  ┆ ---                       │
│ str                        ┆ str              ┆ str                  ┆ str                       │
╞════════════════════════════╪══════════════════╪══════════════════════╪═══════════════════════════╡
│ D1 Relationship discovery  ┆ **3.5**          ┆ **3.5**              ┆ TPR 1.0 on main for both  │
│                            ┆                  ┆                      ┆ runs;…                    │
│ D3 Explanation             ┆ 3.5              ┆ **4.0**              ┆ both reports rank effects │
│                            ┆                  ┆                      ┆ and …                     │
│ D4 Importance              ┆ 3.0              ┆ **3.5**              ┆ planted-effect interval   │
│ quantification             ┆                  ┆                      ┆ covera…                   │
│ D5 Iteration speed (proxy) ┆ **4.0**          ┆ **4.0**              ┆ median logged fit time on │
│                            ┆                  ┆                      ┆ main…                     │
│ D6 Robustness              ┆ **3.0**          ┆ **3.0**              ┆ recovery at n = 200 and   │
│                            ┆                  ┆                      ┆ on mes…                   │
│ D8 Agent-friendliness      ┆ 4.0              ┆ not applicable       ┆ run A completed with no   │
│                            ┆                  ┆                      ┆ human …                   │
│ D9 Uncertainty honesty     ┆ **3.5**          ┆ 3.0                  ┆ coverage near 0.90 for    │
│                            ┆                  ┆                      ┆ both; w…                  │
│ D10 Reproducibility        ┆ **4.0**          ┆ **4.0**              ┆ fresh-directory rerun     │
│                            ┆                  ┆                      ┆ gives by…                 │
│ D2 Causal humility         ┆ not measured     ┆ not measured         ┆ not measured: the battery │
│                            ┆                  ┆                      ┆ has …                     │
│ D7 Human-friendliness      ┆ not measured     ┆ not measured         ┆ not measured: no          │
│                            ┆                  ┆                      ┆ task-based st…            │
└────────────────────────────┴──────────────────┴──────────────────────┴───────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10b1d1dc0>, _boxhead=Boxhead([ColInfo(var='Dimension', type=<ColInfoTypeEnum.default: 1>, column_label='Dimension', column_align='left', column_width=None), ColInfo(var='mboostrs (run A)', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs (run A)', column_align='left', column_width=None), ColInfo(var='Free library (run B)', type=<ColInfoTypeEnum.default: 1>, column_label='Free library (run B)', column_align='left', column_width=None), ColInfo(var='Evidence', type=<ColInfoTypeEnum.default: 1>, column_label='Evidence', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b8354c0>, _spanners=Spanners([]), _heading=Heading(title='Scorecard v1 scores for mboostrs and a free-library analyst', subtitle='Scores on the 1 to 5 anchored scale (higher is better); D5 is a proxy time', preheader=None), _stubhead='Dimension', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10b80f740>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10ade4d70>, _source_notes=['Scores are judged against the anchors in scorecard.md from the reports, deliverables and measured metrics. Run A: mboostrs only. Run B: any library except mboostrs. Source: docs/scorecard/study/SCORES.md; metrics from score.py.'], _footnotes=[], _styles=[], _loc

The weighted profile for run A, using the fully automated weights, is 3.6 over the
eight scored dimensions. Run B has no weighted profile here. The study judged run B
as a reference point, not as a library being scored, so the card's weights apply to
run A only.

## Planted effects and interval coverage

A 90% interval for an effect covers the truth when the interval contains the
planted contrast. The contrast is the average change in the predicted response when
an input moves from its 25th to its 75th percentile, with the other inputs held at
their observed values. The truth is computed from the generator's mean function on
the same training rows. The inputs $x_6$ and $x_7$ are not scored on contrast,
because they have no main effect.

In [4]:
truth = json.loads((STUDY / "truth.json").read_text())["datasets"]["main"]["contrasts"]
effects = {
    run: json.loads((STUDY / "runs" / run / "findings_main.json").read_text())["effects"]
    for run in ["run_A", "run_B"]
}

planted = ["x1", "x2", "x4", "x5"]
planted_rows = []
for name in planted:
    entry = {"Input": name, "Truth": truth[name]}
    for run, label in [("run_A", "mboostrs"), ("run_B", "Free library")]:
        e = effects[run][name]
        covered = e["lower90"] <= truth[name] <= e["upper90"]
        entry[f"{label} estimate"] = e["estimate"]
        entry[f"{label} 90% interval"] = f"[{e['lower90']:.2f}, {e['upper90']:.2f}]"
        entry[f"{label} covers"] = "yes" if covered else "**no**"
    planted_rows.append(entry)

planted_table = pl.DataFrame(planted_rows)
(
    GT(planted_table)
    .tab_header(
        title="Planted main effects on the main dataset: estimates and 90% intervals",
        subtitle="Contrast in response units; 25th to 75th percentile; n = 2000 training rows",
    )
    .fmt_number(columns=["Truth", "mboostrs estimate", "Free library estimate"], decimals=2)
    .fmt_markdown(columns=["mboostrs covers", "Free library covers"])
    .tab_stubhead(label="Input")
    .tab_source_note("Coverage is 'yes' when the 90% interval contains the planted contrast. Source: findings_main.json in each run.")
)

GT(_tbl_data=shape: (4, 8)
┌───────┬──────────┬──────────┬──────────┬──────────┬──────────────┬──────────────────┬─────────┐
│ Input ┆ Truth    ┆ mboostrs ┆ mboostrs ┆ mboostrs ┆ Free library ┆ Free library 90% ┆ Free    │
│ ---   ┆ ---      ┆ estimate ┆ 90%      ┆ covers   ┆ estimate     ┆ interval         ┆ library │
│ str   ┆ f64      ┆ ---      ┆ interval ┆ ---      ┆ ---          ┆ ---              ┆ covers  │
│       ┆          ┆ f64      ┆ ---      ┆ str      ┆ f64          ┆ str              ┆ ---     │
│       ┆          ┆          ┆ str      ┆          ┆              ┆                  ┆ str     │
╞═══════╪══════════╪══════════╪══════════╪══════════╪══════════════╪══════════════════╪═════════╡
│ x1    ┆ 1.353652 ┆ 1.339331 ┆ [1.31,   ┆ yes      ┆ 1.291502     ┆ [1.22, 1.36]     ┆ yes     │
│       ┆          ┆          ┆ 1.37]    ┆          ┆              ┆                  ┆         │
│ x2    ┆ 0.6551   ┆ 0.611539 ┆ [0.58,   ┆ **no**   ┆ 0.627372     ┆ [0.55, 0.70]     ┆ yes     │
│       ┆          ┆          ┆ 0.64]    ┆          ┆              ┆                  ┆         │
│ x4    ┆ 3.983589 ┆ 3.876901 ┆ [3.77,   ┆ **no**   ┆ 3.934606     ┆ [3.85, 4.02]     ┆ yes     │
│       ┆          ┆          ┆ 3.98]    ┆          ┆              ┆                  ┆         │
│ x5    ┆ 1.5      ┆ 1.404325 ┆ [1.30,   ┆ yes      ┆ 1.620677     ┆ [1.54, 1.70]     ┆ **no**  │
│       ┆          ┆          ┆ 1.50]    ┆          ┆              ┆                  ┆         │
└───────┴──────────┴──────────┴──────────┴──────────┴──────────────┴──────────────────┴─────────┘, _body=<great_tables._gt_data.Body object at 0x10ae3ab70>, _boxhead=Boxhead([ColInfo(var='Input', type=<ColInfoTypeEnum.default: 1>, column_label='Input', column_align='left', column_width=None), ColInfo(var='Truth', type=<ColInfoTypeEnum.default: 1>, column_label='Truth', column_align='right', column_width=None), ColInfo(var='mboostrs estimate', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs estimate', column_align='right', column_width=None), ColInfo(var='mboostrs 90% interval', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs 90% interval', column_align='left', column_width=None), ColInfo(var='mboostrs covers', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs covers', column_align='left', column_width=None), ColInfo(var='Free library estimate', type=<ColInfoTypeEnum.default: 1>, column_label='Free library estimate', column_align='right', column_width=None), ColInfo(var='Free library 90% interval', type=<ColInfoTypeEnum.default: 1>, column_label='Free library 90% interval', column_align='left', column_width=None), ColInfo(var='Free library covers', type=<ColInfoTypeEnum.default: 1>, column_label='Free library covers', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a666e70>, _spanners=Spanners([]), _heading=Heading(title='Planted main effects on the main dataset: estimates and 90% intervals', subtitle='Contrast in response units; 25th to 75th percentile; n = 2000 training rows', preheader=None), _stubhead='Input', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a04f440>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10ab82e70>, _source_notes=["Coverage is 'yes' when the 90% interval contains the planted contrast. Source: findings_main.json in each run."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b846d80>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10b834980>, <great_tables._gt_data.FormatInfo object at 0x10b846ed0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), tabl

Run A covers two of the four planted effects. Its intervals for $x_2$ and $x_4$ lie
below the truth. Run B covers three of four, and its one miss is $x_5$, whose
interval lies above the step size of 1.5.

Across all eighteen scored inputs, run A's coverage is 0.89 and run B's is 0.94.
Run A's figure includes fourteen null inputs whose intervals have zero width. Those
intervals are placed at zero by convention, so they cover the truth of zero
trivially. Coverage on the four planted effects is the more informative figure.

In [5]:
coverage = pl.DataFrame(
    [
        {
            "Run": "mboostrs (run A)",
            "Scored inputs": measured["A"]["D4_main"]["n_inputs_scored"],
            "Coverage, all scored inputs": measured["A"]["D4_main"]["contrast_coverage90"],
            "Median width, planted inputs": measured["A"]["D4_main"]["median_width_planted"],
        },
        {
            "Run": "Free library (run B)",
            "Scored inputs": measured["B"]["D4_main"]["n_inputs_scored"],
            "Coverage, all scored inputs": measured["B"]["D4_main"]["contrast_coverage90"],
            "Median width, planted inputs": measured["B"]["D4_main"]["median_width_planted"],
        },
    ]
)
(
    GT(coverage)
    .tab_header(
        title="Contrast interval coverage on the main dataset",
        subtitle="Target coverage 0.90 (higher is closer to nominal); width in response units (lower is tighter)",
    )
    .fmt_number(columns=["Coverage, all scored inputs", "Median width, planted inputs"], decimals=2)
    .tab_stubhead(label="Run")
    .tab_source_note(
        "Coverage over the 18 inputs with a contrast truth (x6 and x7 excluded). Run A's zero-width null "
        "intervals cover trivially; see the text above. Source: score.py on each run."
    )
)

GT(_tbl_data=shape: (2, 4)
┌──────────────────────┬───────────────┬─────────────────────────────┬───────────────────────┐
│ Run                  ┆ Scored inputs ┆ Coverage, all scored inputs ┆ Median width, planted │
│ ---                  ┆ ---           ┆ ---                         ┆ inputs                │
│ str                  ┆ i64           ┆ f64                         ┆ ---                   │
│                      ┆               ┆                             ┆ f64                   │
╞══════════════════════╪═══════════════╪═════════════════════════════╪═══════════════════════╡
│ mboostrs (run A)     ┆ 18            ┆ 0.888889                    ┆ 0.131569              │
│ Free library (run B) ┆ 18            ┆ 0.944444                    ┆ 0.155411              │
└──────────────────────┴───────────────┴─────────────────────────────┴───────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10b835430>, _boxhead=Boxhead([ColInfo(var='Run', type=<ColInfoTypeEnum.default: 1>, column_label='Run', column_align='left', column_width=None), ColInfo(var='Scored inputs', type=<ColInfoTypeEnum.default: 1>, column_label='Scored inputs', column_align='right', column_width=None), ColInfo(var='Coverage, all scored inputs', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage, all scored inputs', column_align='right', column_width=None), ColInfo(var='Median width, planted inputs', type=<ColInfoTypeEnum.default: 1>, column_label='Median width, planted inputs', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b837080>, _spanners=Spanners([]), _heading=Heading(title='Contrast interval coverage on the main dataset', subtitle='Target coverage 0.90 (higher is closer to nominal); width in response units (lower is tighter)', preheader=None), _stubhead='Run', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10b846d20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10b847410>, _source_notes=["Coverage over the 18 inputs with a contrast truth (x6 and x7 excluded). Run A's zero-width null intervals cover trivially; see the text above. Source: score.py on each run."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b844230>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10afdae70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=Options

## Recovery of planted structure

The discovery metrics count the four planted main effects and the $x_6 x_7$ pair.
The true positive rate (TPR) is the share of these five structures that a run
selected. The false discovery proportion (FDP) is the share of selected items that
are not planted. Run A's findings file marks $x_6$ and $x_7$ as selected inputs,
although its report says they enter only through the interaction. The strict
reading, used by `score.py`, counts them as false positives. The table gives both
readings for run A.

In [6]:
def recovery_row(label, dataset, d1, interaction_read_fdp):
    """One row of recovery metrics. Run A's x6 and x7 are interaction members only."""
    return {
        "Run": label,
        "Dataset": dataset,
        "TPR": d1["tpr"],
        "FDP, strict": d1["fdr"],
        "FDP, interaction read": interaction_read_fdp,
    }


# score.py stores main at the top level and small and messy under D6_*.
recovery = pl.DataFrame(
    [
        recovery_row("mboostrs (run A)", "main", measured["A"]["D1_main"], 0.0),
        recovery_row("mboostrs (run A)", "small", measured["A"]["D6_small"]["D1"], 0.0),
        recovery_row("mboostrs (run A)", "messy", measured["A"]["D6_messy"]["D1"], 0.0),
        recovery_row("Free library (run B)", "main", measured["B"]["D1_main"], measured["B"]["D1_main"]["fdr"]),
        recovery_row("Free library (run B)", "small", measured["B"]["D6_small"]["D1"], measured["B"]["D6_small"]["D1"]["fdr"]),
        recovery_row("Free library (run B)", "messy", measured["B"]["D6_messy"]["D1"], measured["B"]["D6_messy"]["D1"]["fdr"]),
    ]
)
(
    GT(recovery)
    .tab_header(
        title="Recovery of the five planted structures",
        subtitle="TPR: share of planted structures selected (higher is better); FDP: share of selections that are not planted (lower is better)",
    )
    .fmt_number(columns=["TPR", "FDP, strict", "FDP, interaction read"], decimals=2)
    .tab_stubhead(label="Run")
    .tab_source_note(
        "Strict FDP counts run A's x6 and x7 as false positives. The interaction-read column counts them as "
        "members of the x6:x7 pair and reports 0 for run A. Datasets: main n = 2000; small n = 200; messy n = 2000 with missing values."
    )
)

GT(_tbl_data=shape: (6, 5)
┌──────────────────────┬─────────┬─────┬─────────────┬───────────────────────┐
│ Run                  ┆ Dataset ┆ TPR ┆ FDP, strict ┆ FDP, interaction read │
│ ---                  ┆ ---     ┆ --- ┆ ---         ┆ ---                   │
│ str                  ┆ str     ┆ f64 ┆ f64         ┆ f64                   │
╞══════════════════════╪═════════╪═════╪═════════════╪═══════════════════════╡
│ mboostrs (run A)     ┆ main    ┆ 1.0 ┆ 0.285714    ┆ 0.0                   │
│ mboostrs (run A)     ┆ small   ┆ 1.0 ┆ 0.285714    ┆ 0.0                   │
│ mboostrs (run A)     ┆ messy   ┆ 1.0 ┆ 0.285714    ┆ 0.0                   │
│ Free library (run B) ┆ main    ┆ 1.0 ┆ 0.0         ┆ 0.0                   │
│ Free library (run B) ┆ small   ┆ 0.8 ┆ 0.0         ┆ 0.0                   │
│ Free library (run B) ┆ messy   ┆ 1.0 ┆ 0.0         ┆ 0.0                   │
└──────────────────────┴─────────┴─────┴─────────────┴───────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10b80f6e0>, _boxhead=Boxhead([ColInfo(var='Run', type=<ColInfoTypeEnum.default: 1>, column_label='Run', column_align='left', column_width=None), ColInfo(var='Dataset', type=<ColInfoTypeEnum.default: 1>, column_label='Dataset', column_align='left', column_width=None), ColInfo(var='TPR', type=<ColInfoTypeEnum.default: 1>, column_label='TPR', column_align='right', column_width=None), ColInfo(var='FDP, strict', type=<ColInfoTypeEnum.default: 1>, column_label='FDP, strict', column_align='right', column_width=None), ColInfo(var='FDP, interaction read', type=<ColInfoTypeEnum.default: 1>, column_label='FDP, interaction read', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b846720>, _spanners=Spanners([]), _heading=Heading(title='Recovery of the five planted structures', subtitle='TPR: share of planted structures selected (higher is better); FDP: share of selections that are not planted (lower is better)', preheader=None), _stubhead='Run', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a667890>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a666e10>, _source_notes=["Strict FDP counts run A's x6 and x7 as false positives. The interaction-read column counts them as members of the x6:x7 pair and reports 0 for run A. Datasets: main n = 2000; small n = 200; messy n = 2000 with missing values."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b835d00>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10a666330>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFF

Run A's recovery is complete on all three datasets, apart from the x6 and x7
labelling question. Run B misses one planted structure at n = 200. The
recovery is not the same thing as a correct effect size, so the interval table
above is the check on magnitudes.

## Predictive intervals

Each run predicted a 90% interval for each new response on the 1000 test rows of
`main`. The table reports the share of responses inside the interval and the rank
correlation between the interval half-width and the noise scale $\sigma(x_8)$.
The correlation tests whether the interval widens where the noise is larger.

In [7]:
predictive = pl.DataFrame(
    [
        {
            "Run": "mboostrs (run A)",
            "Coverage, nominal 0.90": measured["A"]["D9_main"]["predictive_coverage90"],
            "Spearman: half-width vs sigma": measured["A"]["D9_main"]["spearman_halfwidth_vs_sigma"],
        },
        {
            "Run": "Free library (run B)",
            "Coverage, nominal 0.90": measured["B"]["D9_main"]["predictive_coverage90"],
            "Spearman: half-width vs sigma": measured["B"]["D9_main"]["spearman_halfwidth_vs_sigma"],
        },
    ]
)
(
    GT(predictive)
    .tab_header(
        title="90% predictive intervals on 1000 new responses",
        subtitle="Coverage near 0.90 is calibrated; a Spearman correlation near 1 means the width tracks the noise scale",
    )
    .fmt_number(columns=["Coverage, nominal 0.90", "Spearman: half-width vs sigma"], decimals=2)
    .tab_stubhead(label="Run")
    .tab_source_note("Test truth: the noise scale sigma(x8) and the realised response, from the generator. Source: predictions_main.csv and score.py.")
)

GT(_tbl_data=shape: (2, 3)
┌──────────────────────┬────────────────────────┬───────────────────────────────┐
│ Run                  ┆ Coverage, nominal 0.90 ┆ Spearman: half-width vs sigma │
│ ---                  ┆ ---                    ┆ ---                           │
│ str                  ┆ f64                    ┆ f64                           │
╞══════════════════════╪════════════════════════╪═══════════════════════════════╡
│ mboostrs (run A)     ┆ 0.89                   ┆ 0.982849                      │
│ Free library (run B) ┆ 0.882                  ┆ 0.584282                      │
└──────────────────────┴────────────────────────┴───────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10a667bf0>, _boxhead=Boxhead([ColInfo(var='Run', type=<ColInfoTypeEnum.default: 1>, column_label='Run', column_align='left', column_width=None), ColInfo(var='Coverage, nominal 0.90', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage, nominal 0.90', column_align='right', column_width=None), ColInfo(var='Spearman: half-width vs sigma', type=<ColInfoTypeEnum.default: 1>, column_label='Spearman: half-width vs sigma', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b845a00>, _spanners=Spanners([]), _heading=Heading(title='90% predictive intervals on 1000 new responses', subtitle='Coverage near 0.90 is calibrated; a Spearman correlation near 1 means the width tracks the noise scale', preheader=None), _stubhead='Run', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10b835310>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10b834c20>, _source_notes=['Test truth: the noise scale sigma(x8) and the realised response, from the generator. Source: predictions_main.csv and score.py.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b835c40>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10b845400>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(sc

Both runs cover close to 0.90. Run A's intervals widen with the noise scale more
closely, with a rank correlation of 0.98 against 0.58 for run B. For small samples,
run A dropped its scale model: at n = 200 its widths reached 953 response units, so
the small-sample intervals come from a different method and are not comparable in
kind.

## Reproducing the runs

The released run directories contain the findings, predictions, fit logs and
`run_all.py` for each analyst. Running `run_all.py` from a fresh directory on the
data reproduces every findings and prediction file byte for byte, for both runs.
The commands are below. They are not executed on this page because the data and
the free-library venv are not in the repository.

```bash
# Regenerate the battery from the seeds recorded in truth.json (writes data and truth)
python docs/scorecard/study/gen.py <data_dir> <truth_dir>

# Score one run against the truth
python docs/scorecard/study/score.py docs/scorecard/study docs/scorecard/study/runs/run_A

# Regenerate one run's deliverables in a fresh directory
python docs/scorecard/study/runs/run_A/run_all.py <data_dir>
```

## Findings

- **mboostrs recovers the planted structure.** Run A selected the four planted main
  effects and the $x_6 x_7$ pair on all three datasets. Its only extra selections are
  x6 and x7, which its report describes as interaction members.
- **Interval calibration on effect sizes is the weak part.** Two of run A's four
  planted intervals miss the truth, and both lie below it. Run B misses one.
- **Predictive intervals are close to nominal and adapt to the noise scale for
  mboostrs.** Coverage is 0.89, and the width tracks sigma with a rank correlation
  of 0.98.
- **The free-library analyst reached similar recovery with hand-built methods.**
  Run B used numpy and scipy for the models, and scikit-learn for imputation on the
  messy data. On main and messy its recovery matched run A's under the interaction
  reading of x6 and x7. It missed one planted structure at n = 200.
- **Both runs reproduce from their own scripts.** The fresh-directory reruns match
  byte for byte.

## Limits

- **One battery, one seed per dataset.** These scores describe this battery only.
- **The x6 and x7 rule was not defined in the protocol.** The protocol said that
  `selected` means a mean effect, and run A's findings file sets `selected` true for
  x6 and x7 while its report says they enter only through the interaction. The
  scores above use the strict reading in the headline table, and the recovery table
  gives both.
- **Zero-width intervals inflate coverage.** Run A reports zero-width intervals for
  inputs it did not select. These cover the truth of zero by construction, so the
  overall coverage figure overstates its calibration.
- **D2 and D7 were not measured,** and D5 is a proxy.
- **Judged scores are one reader's judgment** against the anchors, from the reports
  and deliverables. The run transcripts were not reviewed line by line.
- **Truth isolation was by instruction.** The analysts were told not to read the
  truth file, and the file was kept outside the repository during the runs. Nothing
  in the sandbox enforced that.
- **The card version.** This study was scored on the ten-dimension version 1 card.
  `docs/scorecard/migration.md` names version 2.1 as the only card for new evaluations.
  Version 2.1 splits D1 and adds D11, so the scores above are version 1 results.